## Cài đặt thư viện theo hướng dẫn của unsloth

In [15]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Cài đặt IntentClassification

In [16]:
import yaml
from unsloth import FastLanguageModel

class IntentClassification:
    def __init__(self, model_path):
        # model_path trỏ tới file cấu hình (inference.yaml)
        with open(model_path, 'r', encoding='utf-8') as f:
            config = yaml.safe_load(f)
        
        # Lấy đường dẫn checkpoint từ file config
        checkpoint_dir = config['checkpoint_path']
        
        # Load mô hình và tokenizer[cite: 2]
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name = checkpoint_dir,
            max_seq_length = config.get('max_seq_length', 256),
            dtype = None,
            load_in_4bit = True,
        )
        
        # Kích hoạt chế độ inference nhanh của Unsloth
        FastLanguageModel.for_inference(self.model)
        
        # Khai báo lại template giống hệt như lúc train
        self.alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert intent classification AI for a bank. Classify the user's input message into the correct banking intent.

### Text:
{}

### Label:
{}"""

    def __call__(self, message):        
        inputs = self.tokenizer(
            [
                self.alpaca_prompt.format(
                    message, # input
                    "", # output - leave this blank for generation!
                )
            ], return_tensors = "pt").to("cuda")
        
        # Sinh text dự đoán
        outputs = self.model.generate(
            **inputs, 
            max_new_tokens = 32,
            use_cache = True,
            pad_token_id = self.tokenizer.eos_token_id
        )
        
        # Decode kết quả từ token ra string
        decoded_output = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        # Cắt chuỗi để chỉ lấy phần nhãn do model sinh ra ở cuối cùng
        predicted_label = decoded_output.split("### Label:\n")[-1].strip()
        
        return predicted_label

## Khởi tạo đối tượng

In [17]:
intent_classifier = IntentClassification(model_path="/kaggle/input/datasets/latorange/banking-intent-unsloth-2/banking-intent-unsloth/configs/inference.yaml")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## Load label_mapping

In [18]:
import json

label_mapping_path = "/kaggle/input/datasets/latorange/banking-intent-unsloth-2/banking-intent-unsloth/configs/label_mapping.json"

with open(label_mapping_path, 'r', encoding='utf-8') as f:
    mapping_data = json.load(f)

id2label = mapping_data['id2label']
label2id = mapping_data['label2id']

## Chạy thử nghiệm

In [20]:
sample_message = input("Write your problem: ")

prediction = intent_classifier(sample_message)

print()
print(f"Message: {sample_message}")
print(f"Predicted Intent ID: {prediction}")
print(f"Predicted Intent Label: {id2label[prediction]}")

Write your problem:  I don't know how to use my credit card.



Message: I don't know how to use my credit card.
Predicted Intent ID: 54
Predicted Intent Label: supported_cards_and_currencies


## Inference trên toàn bộ tập test

In [14]:
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

test_data_path = "/kaggle/input/datasets/latorange/banking-intent-unsloth-2/banking-intent-unsloth/sample_data/test.csv" 
print(f"Đọc dữ liệu từ {test_data_path}...")
test_df = pd.read_csv(test_data_path)

# Khởi tạo 2 list để lưu kết quả
true_labels = []
predicted_labels = []

print("Bắt đầu chạy inference trên tập test...")
for index, row in tqdm(test_df.iterrows(), total=test_df.shape[0]):
    text = str(row['text'])

    true_label = str(row['label'])
    true_labels.append(true_label)

    pred_label = intent_classifier(text)
    predicted_labels.append(pred_label)

# 4. Tính toán và in ra độ chính xác
accuracy = accuracy_score(true_labels, predicted_labels)
print("\n" + "="*50)
print(f"ĐỘ CHÍNH XÁC (ACCURACY) TRÊN TẬP TEST: {accuracy * 100:.2f}%")
print("="*50)

# (Tùy chọn) In ra báo cáo chi tiết để xem model dự đoán tốt/kém ở những intent nào
print("\nBÁO CÁO CHI TIẾT (Classification Report):")
print(classification_report(true_labels, predicted_labels, zero_division=0))

Đọc dữ liệu từ /kaggle/input/datasets/latorange/banking-intent-unsloth-2/banking-intent-unsloth/sample_data/test.csv...
Bắt đầu chạy inference trên tập test...


100%|██████████| 770/770 [03:57<00:00,  3.25it/s]


ĐỘ CHÍNH XÁC (ACCURACY) TRÊN TẬP TEST: 90.13%

BÁO CÁO CHI TIẾT (Classification Report):
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       0.91      1.00      0.95        10
          10       1.00      0.90      0.95        10
          11       0.90      0.90      0.90        10
          12       0.90      0.90      0.90        10
          13       1.00      0.90      0.95        10
          14       1.00      0.80      0.89        10
          15       0.82      0.90      0.86        10
          16       1.00      0.90      0.95        10
          17       1.00      0.90      0.95        10
          18       0.90      0.90      0.90        10
          19       1.00      1.00      1.00        10
           2       1.00      1.00      1.00        10
          20       1.00      0.90      0.95        10
          21       1.00      1.00      1.00        10
          22       0.91      1.00      0.95  